# Vnstock — Vietnam market data for prediction

Use the same **Python kernel** as your project (`.venv`).

- **Live quotes**: `stock.trading.price_board(...)` (intraday board; not used as model input).
- **Historical OHLCV**: `fetch_vnstock` in `src/data/loaders.py` wraps `stock.quote.history(...)` and returns the same columns as the Nasdaq path expects for `add_all_features` (`open`, `high`, `low`, `close`, `volume`; no `adj_close` — see `compute_target_price` in `preprocess.py`).

With **vnstock 3.5.x**, `source="VCI"` often fails during client init; **`KBS`** is the reliable default for daily history in this environment.

In [1]:
import sys
from pathlib import Path

# Project root (parent of notebooks/)
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import vnstock

📦 **Vnstock 4.0.2 is available**
Current: 3.5.1 (Python 3.13 (venv))
Update: `/Users/cps/DL4AI-240166-project-1/.venv/bin/python -m pip install vnstock --upgrade`
Release: https://vnstocks.com/docs/tai-lieu/lich-su-phien-ban

📦 **Vnai 2.4.8 is available**
Current: 2.4.6 (Python 3.13 (venv))
Update: `/Users/cps/DL4AI-240166-project-1/.venv/bin/python -m pip install vnai --upgrade`
Release: https://pypi.org/project/vnai/#history

In [2]:
# vnstock 3.x may not expose __version__ on the module — use metadata instead
from importlib.metadata import version, PackageNotFoundError

try:
    print("vnstock version:", version("vnstock"))
except PackageNotFoundError:
    print("vnstock not installed in this kernel — pick .venv interpreter")

vnstock version: 3.5.1


In [7]:
from vnstock import Vnstock

stock = Vnstock().stock(symbol='FPT', source='KBS')

# gọi trading qua object
df_board = stock.trading.price_board(['FPT'])

print(df_board.head())


⚠️ 
⚠️  GIỚI HẠN API ĐÃ ĐẠT TỐI ĐA (Rate Limit Exceeded)

📌 Bạn đã đạt giới hạn tối đa số lượt yêu cầu API trong 1 phút (minute).
   (You have reached the maximum API request limit for this period)

📊 Chi tiết (Details):
   • Gói hiện tại: Khách (Guest)
   • Giới hạn: 20 requests/phút
   • Đã sử dụng: 20/20
   • Chờ 14 giây để tiếp tục (Wait to retry)

💡 Giải pháp (Solutions):
   1️⃣ Chờ 14 giây rồi thử lại
      (Wait and retry)
   2️⃣ Tham gia chương trình tài trợ dự án (Sponsor) để sử dụng không gián đoạn.
      Lưu ý: vnstock là công cụ mã nguồn mở giúp tự động hoá kết nối với
      các API công khai mà bạn vốn đã có quyền truy cập hợp lệ, không
      phải nhà cung cấp dữ liệu. Việc tài trợ giúp duy trì
      nghiên cứu - phát triển công cụ và hạ tầng công nghệ phục vụ cộng đồng.

🚀 Nâng cấp (Upgrade):
   • Phiên bản cộng đồng (60 request/phút - Community):
     Đăng ký API key miễn phí: https://vnstocks.com/login
   • Gói thành viên tài trợ (180-600 request/phút - Sponsor):
     

SystemExit: Rate limit exceeded. 
============================================================
⚠️  GIỚI HẠN API ĐÃ ĐẠT TỐI ĐA (Rate Limit Exceeded)
============================================================

📌 Bạn đã đạt giới hạn tối đa số lượt yêu cầu API trong 1 phút (minute).
   (You have reached the maximum API request limit for this period)

📊 Chi tiết (Details):
   • Gói hiện tại: Khách (Guest)
   • Giới hạn: 20 requests/phút
   • Đã sử dụng: 20/20
   • Chờ 14 giây để tiếp tục (Wait to retry)

💡 Giải pháp (Solutions):
   1️⃣ Chờ 14 giây rồi thử lại
      (Wait and retry)
   2️⃣ Tham gia chương trình tài trợ dự án (Sponsor) để sử dụng không gián đoạn.
      Lưu ý: vnstock là công cụ mã nguồn mở giúp tự động hoá kết nối với
      các API công khai mà bạn vốn đã có quyền truy cập hợp lệ, không
      phải nhà cung cấp dữ liệu. Việc tài trợ giúp duy trì
      nghiên cứu - phát triển công cụ và hạ tầng công nghệ phục vụ cộng đồng.

🚀 Nâng cấp (Upgrade):
   • Phiên bản cộng đồng (60 r

In [4]:
import time

import pandas as pd

from src.data.loaders import fetch_vnstock

OUT_DIR = ROOT / "notebooks" / "data" / "vietnam" / "csv"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# HOSE names with decent liquidity — edit freely
TICKERS = ["FPT", "VCB", "VHM", "VNM", "HPG"]
START = "2020-01-01"
END = pd.Timestamp.today().strftime("%Y-%m-%d")
SOURCE = "KBS"

frames = []
for sym in TICKERS:
    try:
        d = fetch_vnstock(sym, START, END, source=SOURCE)
        d = d.assign(ticker=sym)
        frames.append(d)
        single = OUT_DIR / f"{sym}_ohlcv.csv"
        d.to_csv(single)
        print(f"{sym}: {len(d)} rows → {single}")
    except Exception as exc:
        print(f"{sym}: ERROR {type(exc).__name__}: {exc}")
    time.sleep(0.35)

if frames:
    panel = pd.concat(frames, axis=0).reset_index().sort_values(["date", "ticker"])
    combined = OUT_DIR / "vietnam_ohlcv_panel.csv"
    panel.to_csv(combined, index=False)
    print(f"Combined panel: {combined}  shape={panel.shape}")
else:
    print("No frames — check network / ticker symbols / vnstock source.")

FPT: 1575 rows → /Users/cps/DL4AI-240166-project-1/notebooks/data/vietnam/csv/FPT_ohlcv.csv
VCB: 1575 rows → /Users/cps/DL4AI-240166-project-1/notebooks/data/vietnam/csv/VCB_ohlcv.csv
VHM: 1575 rows → /Users/cps/DL4AI-240166-project-1/notebooks/data/vietnam/csv/VHM_ohlcv.csv
VNM: 1575 rows → /Users/cps/DL4AI-240166-project-1/notebooks/data/vietnam/csv/VNM_ohlcv.csv
HPG: 1575 rows → /Users/cps/DL4AI-240166-project-1/notebooks/data/vietnam/csv/HPG_ohlcv.csv
Combined panel: /Users/cps/DL4AI-240166-project-1/notebooks/data/vietnam/csv/vietnam_ohlcv_panel.csv  shape=(7875, 7)


In [35]:
import time

import pandas as pd

from src.data.loaders import fetch_vnstock

OUT_DIR = ROOT / "notebooks" / "data" / "vietnam" / "csv"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# HOSE names with decent liquidity — edit freely
TICKERS = {
    'VIC',
    'TCB',
    'MSN',
    'MWG',
    'VND'
}
START = "2020-01-01"
END = pd.Timestamp.today().strftime("%Y-%m-%d")
SOURCE = "KBS"

frames = []
for sym in TICKERS:
    try:
        d = fetch_vnstock(sym, START, END, source=SOURCE)
        d = d.assign(ticker=sym)
        frames.append(d)
        single = OUT_DIR / f"{sym}_ohlcv.csv"
        d.to_csv(single)
        print(f"{sym}: {len(d)} rows → {single}")
    except Exception as exc:
        print(f"{sym}: ERROR {type(exc).__name__}: {exc}")
    time.sleep(0.35)

if frames:
    panel = pd.concat(frames, axis=0).reset_index().sort_values(["date", "ticker"])
    combined = OUT_DIR / "vietnam_ohlcv_panel.csv"
    panel.to_csv(combined, index=False)
    print(f"Combined panel: {combined}  shape={panel.shape}")
else:
    print("No frames — check network / ticker symbols / vnstock source.")

MSN: 1575 rows → /Users/cps/DL4AI-240166-project-1/notebooks/data/vietnam/csv/MSN_ohlcv.csv
TCB: 1575 rows → /Users/cps/DL4AI-240166-project-1/notebooks/data/vietnam/csv/TCB_ohlcv.csv
VIC: 1575 rows → /Users/cps/DL4AI-240166-project-1/notebooks/data/vietnam/csv/VIC_ohlcv.csv
VND: 1569 rows → /Users/cps/DL4AI-240166-project-1/notebooks/data/vietnam/csv/VND_ohlcv.csv
MWG: 1575 rows → /Users/cps/DL4AI-240166-project-1/notebooks/data/vietnam/csv/MWG_ohlcv.csv
Combined panel: /Users/cps/DL4AI-240166-project-1/notebooks/data/vietnam/csv/vietnam_ohlcv_panel.csv  shape=(7869, 7)


In [6]:
import time

import pandas as pd

from src.data.loaders import fetch_vnstock

OUT_DIR = ROOT / "notebooks" / "data" / "vietnam" / "csv"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# --- THE EXPANDED TICKER LIST (30 TOTAL) ---
TICKERS = [
    # Original 10
    # 20 NEW TICKERS
    "BID", "CTG", "MBB", "ACB", "VPB", "HDB", "STB", # Banking
    "BVH",                                           # Insurance/Finance
    "NVL", "PDR", "KDH",                            # Real Estate
    "HSG", "GAS", "PLX", "PVD",                     # Steel/Industrial
    "PNJ", "SAB", "MCH",                            # Consumer/Retail
    "VGI",                                          # Telecom
    "ACV"                                           # Airport
]
START = "2020-01-01"
END = "2026-04-28"
SOURCE = "KBS"

frames = []
for sym in TICKERS:
    try:
        d = fetch_vnstock(sym, START, END, source=SOURCE)
        d = d.assign(ticker=sym)
        frames.append(d)
        single = OUT_DIR / f"{sym}_ohlcv.csv"
        d.to_csv(single)
        print(f"{sym}: {len(d)} rows → {single}")
    except Exception as exc:
        print(f"{sym}: ERROR {type(exc).__name__}: {exc}")
    time.sleep(0.35)

if frames:
    panel = pd.concat(frames, axis=0).reset_index().sort_values(["date", "ticker"])
    combined = OUT_DIR / "vietnam_ohlcv_panel.csv"
    panel.to_csv(combined, index=False)
    print(f"Combined panel: {combined}  shape={panel.shape}")
else:
    print("No frames — check network / ticker symbols / vnstock source.")

SystemExit: Rate limit exceeded. 
============================================================
⚠️  GIỚI HẠN API ĐÃ ĐẠT TỐI ĐA (Rate Limit Exceeded)
============================================================

📌 Bạn đã đạt giới hạn tối đa số lượt yêu cầu API trong 1 phút (minute).
   (You have reached the maximum API request limit for this period)

📊 Chi tiết (Details):
   • Gói hiện tại: Khách (Guest)
   • Giới hạn: 20 requests/phút
   • Đã sử dụng: 20/20
   • Chờ 18 giây để tiếp tục (Wait to retry)

💡 Giải pháp (Solutions):
   1️⃣ Chờ 18 giây rồi thử lại
      (Wait and retry)
   2️⃣ Tham gia chương trình tài trợ dự án (Sponsor) để sử dụng không gián đoạn.
      Lưu ý: vnstock là công cụ mã nguồn mở giúp tự động hoá kết nối với
      các API công khai mà bạn vốn đã có quyền truy cập hợp lệ, không
      phải nhà cung cấp dữ liệu. Việc tài trợ giúp duy trì
      nghiên cứu - phát triển công cụ và hạ tầng công nghệ phục vụ cộng đồng.

🚀 Nâng cấp (Upgrade):
   • Phiên bản cộng đồng (60 r

### Crawl Data from Yahoo finance

In [8]:
import time
import pandas as pd
import yfinance as yf
from pathlib import Path

# Setup paths - adjusting for your ROOT variable or using local path
ROOT = Path.cwd()
OUT_DIR = ROOT / "notebooks" / "data" / "vietnam" / "csv"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# --- THE EXPANDED TICKER LIST ---
# Appending .VN for HOSE (Most of these are on HOSE)
TICKERS = [
    "BID", "CTG", "MBB", "ACB", "VPB", "HDB", "STB", 
    "BVH", "NVL", "PDR", "KDH", "HSG", "GAS", "PLX", 
    "PVD", "PNJ", "SAB", "MCH", "VGI", "ACV"
]

# Map tickers to Yahoo Finance format (e.g., VGI and ACV are UPCoM, others are HOSE)
YF_TICKERS = [f"{t}.VN" if t not in ["VGI", "ACV", "MCH"] else f"{t}.HN" for t in TICKERS]

START = "2020-01-01"
END = "2026-04-28"

frames = []

for original_sym, yf_sym in zip(TICKERS, YF_TICKERS):
    try:
        # Fetching data
        df = yf.download(yf_sym, start=START, end=END, progress=False)
        
        if df.empty:
            print(f"{original_sym}: No data found.")
            continue

        # Standardizing format: Reset index to get 'Date' as a column
        df = df.reset_index()
        
        # Rename columns to standard OHLCV if necessary
        df.columns = [col.lower() if isinstance(col, str) else col[0].lower() for col in df.columns]
        df = df.rename(columns={'adj close': 'adj_close'})
        
        # Add the ticker column
        df['ticker'] = original_sym
        frames.append(df)
        
        # Save individual CSV
        single_path = OUT_DIR / f"{original_sym}_ohlcv.csv"
        df.to_csv(single_path, index=False)
        print(f"{original_sym}: {len(df)} rows → {single_path}")
        
    except Exception as exc:
        print(f"{original_sym}: ERROR {type(exc).__name__}: {exc}")
    
    # yfinance is more robust, but a tiny sleep prevents IP blocks
    time.sleep(0.2)

if frames:
    # Combine into a single panel
    panel = pd.concat(frames, axis=0).sort_values(["date", "ticker"])
    combined_path = OUT_DIR / "vietnam_ohlcv_panel.csv"
    panel.to_csv(combined_path, index=False)
    print(f"\n✅ Combined panel saved: {combined_path}")
    print(f"Final shape: {panel.shape}")
else:
    print("\n❌ No data collected. Check ticker suffixes or network connection.")

BID: 1574 rows → /Users/cps/DL4AI-240166-project-1/notebooks/notebooks/data/vietnam/csv/BID_ohlcv.csv
CTG: 1574 rows → /Users/cps/DL4AI-240166-project-1/notebooks/notebooks/data/vietnam/csv/CTG_ohlcv.csv
MBB: 1574 rows → /Users/cps/DL4AI-240166-project-1/notebooks/notebooks/data/vietnam/csv/MBB_ohlcv.csv
ACB: 1569 rows → /Users/cps/DL4AI-240166-project-1/notebooks/notebooks/data/vietnam/csv/ACB_ohlcv.csv
VPB: 682 rows → /Users/cps/DL4AI-240166-project-1/notebooks/notebooks/data/vietnam/csv/VPB_ohlcv.csv
HDB: 1574 rows → /Users/cps/DL4AI-240166-project-1/notebooks/notebooks/data/vietnam/csv/HDB_ohlcv.csv
STB: 681 rows → /Users/cps/DL4AI-240166-project-1/notebooks/notebooks/data/vietnam/csv/STB_ohlcv.csv
BVH: 681 rows → /Users/cps/DL4AI-240166-project-1/notebooks/notebooks/data/vietnam/csv/BVH_ohlcv.csv
NVL: 682 rows → /Users/cps/DL4AI-240166-project-1/notebooks/notebooks/data/vietnam/csv/NVL_ohlcv.csv
PDR: 1574 rows → /Users/cps/DL4AI-240166-project-1/notebooks/notebooks/data/vietnam/cs

HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: MCH.HN"}}}
$MCH.HN: possibly delisted; no timezone found

1 Failed download:
['MCH.HN']: possibly delisted; no timezone found


MCH: No data found.


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: VGI.HN"}}}
$VGI.HN: possibly delisted; no timezone found

1 Failed download:
['VGI.HN']: possibly delisted; no timezone found


VGI: No data found.


$ACV.HN: possibly delisted; no timezone found

1 Failed download:
['ACV.HN']: possibly delisted; no timezone found


ACV: No data found.

✅ Combined panel saved: /Users/cps/DL4AI-240166-project-1/notebooks/notebooks/data/vietnam/csv/vietnam_ohlcv_panel.csv
Final shape: (21392, 7)


In [9]:
import time
import pandas as pd
import yfinance as yf
from pathlib import Path

# Setup paths
ROOT = Path.cwd()
OUT_DIR = ROOT / "notebooks" / "data" / "vietnam" / "csv"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# --- 20 NEW TICKERS (Excluding your existing 10) ---
TICKERS = [
    "BID", "CTG", "MBB", "ACB", "VPB", "HDB", "STB", "TPB", "SHB", # Banking
    "NVL", "PDR", "KDH", "NLG", "DXG",                            # Real Estate
    "GAS", "PLX", "PVD", "HSG",                                   # Energy/Steel
    "PNJ", "SAB"                                                  # Retail/Consumer
]

# Standardize for Yahoo Finance
YF_TICKERS = [f"{t}.VN" for t in TICKERS]

START = "2020-01-01"
END = "2026-04-28"

frames = []

print(f"🚀 Starting download for {len(TICKERS)} new tickers...\n")

for original_sym, yf_sym in zip(TICKERS, YF_TICKERS):
    try:
        df = yf.download(yf_sym, start=START, end=END, progress=False)
        
        if df.empty:
            print(f"⚠️ {original_sym}: No data found.")
            continue

        # Standardizing format
        df = df.reset_index()
        
        # Clean column names (handles potential MultiIndex)
        df.columns = [
            col[0].lower() if isinstance(col, tuple) else col.lower() 
            for col in df.columns
        ]
        
        df = df.rename(columns={'adj close': 'adj_close'})
        df['ticker'] = original_sym
        
        # Save individual CSV
        single_path = OUT_DIR / f"{original_sym}_ohlcv.csv"
        df.to_csv(single_path, index=False)
        
        frames.append(df)
        print(f"✅ {original_sym}: {len(df)} rows")
        
    except Exception as exc:
        print(f"❌ {original_sym}: Error -> {exc}")
    
    time.sleep(0.5) 

if frames:
    # Combine into a single panel
    panel = pd.concat(frames, axis=0).sort_values(["date", "ticker"])
    combined_path = OUT_DIR / "vietnam_ohlcv_new_20_panel.csv"
    panel.to_csv(combined_path, index=False)
    print(f"\n--- FINISHED ---")
    print(f"New panel saved: {combined_path}")
    print(f"Total Tickers Added: {len(frames)}")

🚀 Starting download for 20 new tickers...

✅ BID: 1574 rows
✅ CTG: 1574 rows
✅ MBB: 1574 rows
✅ ACB: 1569 rows
✅ VPB: 682 rows
✅ HDB: 1574 rows
✅ STB: 681 rows
✅ TPB: 1574 rows
✅ SHB: 1571 rows
✅ NVL: 682 rows
✅ PDR: 1574 rows
✅ KDH: 1574 rows
✅ NLG: 681 rows
✅ DXG: 1574 rows
✅ GAS: 1574 rows
✅ PLX: 681 rows
✅ PVD: 681 rows
✅ HSG: 1574 rows
✅ PNJ: 1574 rows
✅ SAB: 1569 rows

--- FINISHED ---
New panel saved: /Users/cps/DL4AI-240166-project-1/notebooks/notebooks/data/vietnam/csv/vietnam_ohlcv_new_20_panel.csv
Total Tickers Added: 20


In [10]:
import time
import pandas as pd
import yfinance as yf
from pathlib import Path

# 1. Setup paths (Matching your directory structure)
ROOT = Path.cwd()
OUT_DIR = ROOT / "notebooks" / "data" / "vietnam" / "csv"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# 2. Define the 20 NEW Tickers (Excluding your existing 10)
TICKERS = [
    "BID", "CTG", "MBB", "ACB", "VPB", "HDB", "STB", "TPB", "SHB", # Banking
    "NVL", "PDR", "KDH", "NLG", "DXG",                            # Real Estate
    "GAS", "PLX", "PVD", "HSG",                                   # Energy/Steel
    "PNJ", "SAB"                                                  # Retail/Consumer
]

START = "2020-01-01"
END = "2026-04-28"

print(f"🚀 Starting individual downloads for {len(TICKERS)} tickers...")

for ticker in TICKERS:
    yf_sym = f"{ticker}.VN"
    try:
        # Download data
        df = yf.download(yf_sym, start=START, end=END, progress=False)
        
        if df.empty:
            print(f"⚠️ {ticker}: No data found.")
            continue

        # Standardizing format to match your existing CSVs
        df = df.reset_index()
        
        # Clean MultiIndex column names if present
        df.columns = [
            col[0].lower() if isinstance(col, tuple) else col.lower() 
            for col in df.columns
        ]
        
        # Standardize specific column names
        df = df.rename(columns={'adj close': 'adj_close'})
        df['ticker'] = ticker
        
        # Save as a separate CSV file
        file_path = OUT_DIR / f"{ticker}_ohlcv.csv"
        df.to_csv(file_path, index=False)
        
        print(f"✅ {ticker}: {len(df)} rows -> {file_path.name}")
        
    except Exception as exc:
        print(f"❌ {ticker}: Failed with error: {exc}")
    
    # Pause briefly to prevent API rate limiting
    time.sleep(0.5)

print("\nProcessing complete. Check your 'vietnam/csv' folder for the new files.")

🚀 Starting individual downloads for 20 tickers...
✅ BID: 1574 rows -> BID_ohlcv.csv
✅ CTG: 1574 rows -> CTG_ohlcv.csv
✅ MBB: 1574 rows -> MBB_ohlcv.csv
✅ ACB: 1569 rows -> ACB_ohlcv.csv
✅ VPB: 682 rows -> VPB_ohlcv.csv
✅ HDB: 1574 rows -> HDB_ohlcv.csv
✅ STB: 681 rows -> STB_ohlcv.csv
✅ TPB: 1574 rows -> TPB_ohlcv.csv
✅ SHB: 1571 rows -> SHB_ohlcv.csv
✅ NVL: 682 rows -> NVL_ohlcv.csv
✅ PDR: 1574 rows -> PDR_ohlcv.csv
✅ KDH: 1574 rows -> KDH_ohlcv.csv
✅ NLG: 681 rows -> NLG_ohlcv.csv
✅ DXG: 1574 rows -> DXG_ohlcv.csv
✅ GAS: 1574 rows -> GAS_ohlcv.csv
✅ PLX: 681 rows -> PLX_ohlcv.csv
✅ PVD: 681 rows -> PVD_ohlcv.csv
✅ HSG: 1574 rows -> HSG_ohlcv.csv
✅ PNJ: 1574 rows -> PNJ_ohlcv.csv
✅ SAB: 1569 rows -> SAB_ohlcv.csv

Processing complete. Check your 'vietnam/csv' folder for the new files.


In [ ]:
BID, CTG,MBB, ACB, HDB,TPB, SHB, PDR, KDH,DXG,GAS,HSG,PNJ,SAB, MBB, HSG, PNJ, 



